# 03 . Evaluate MEG-Mamba features

Same protocol as every other model in this project, so the numbers are comparable.
Stratified 5-fold, seed 123, logistic regression, pooled out-of-fold metrics, and a
paired bootstrap against BrainOmni on the same subjects.

Needs `mamba_features.npy` back from Colab in `results_mamba/`.

In [3]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

OUT, PARCEL = '../results_mamba', '../results_parcel'
F = f'{OUT}/mamba_features.npy'
assert os.path.exists(F), 'mamba_features.npy missing, run the Colab notebook first'

X = np.load(F)
y = np.load(f'{OUT}/labels_mamba.npy')
assert len(X) == len(y), f'{len(X)} features vs {len(y)} labels'
print(f'features {X.shape}   AD {int(y.sum())}   control {int((y==0).sum())}')
print(f'majority-class baseline {max(np.bincount(y))/len(y):.3f}')

cv = StratifiedKFold(5, shuffle=True, random_state=123)

features (161, 25600)   AD 77   control 84
majority-class baseline 0.522


In [4]:
# Note: these features are ~25600-dim for 161 subjects, six times wider than the
# BrainOmni ones. Logistic regression is carrying that entirely through
# regularisation. If results look unstable, PCA to ~50 components first and
# compare, as notebooks_parcel/08 did.
r = cross_validate(LogisticRegression(max_iter=2000), X, y, cv=cv,
                   scoring=['accuracy', 'roc_auc'])
print('MEG-Mamba frozen + LR, 161 subjects')
print(f'  accuracy {r["test_accuracy"].mean():.3f}   per fold {np.round(r["test_accuracy"],3)}')
print(f'  AUC      {r["test_roc_auc"].mean():.3f}   per fold {np.round(r["test_roc_auc"],3)}')

MEG-Mamba frozen + LR, 161 subjects
  accuracy 0.826   per fold [0.909 0.688 0.875 0.844 0.812]
  AUC      0.898   per fold [0.982 0.793 0.941 0.882 0.89 ]


In [5]:
def oof(Xs, ys):
    """Pooled out-of-fold predictions, the convention used throughout this project."""
    p, q = np.zeros(len(ys), int), np.zeros(len(ys))
    for tr, te in cv.split(Xs, ys):
        m = LogisticRegression(max_iter=2000).fit(Xs[tr], ys[tr])
        p[te], q[te] = m.predict(Xs[te]), m.predict_proba(Xs[te])[:, 1]
    return p, q

def boot(p, q, ys, nb=2000, seed=123):
    rng, A, U = np.random.default_rng(seed), [], []
    for _ in range(nb):
        b = rng.choice(len(ys), len(ys), replace=True)
        if len(np.unique(ys[b])) < 2:
            continue
        A.append(accuracy_score(ys[b], p[b])); U.append(roc_auc_score(ys[b], q[b]))
    return np.percentile(A, [2.5, 97.5]), np.percentile(U, [2.5, 97.5])

pm, qm = oof(X, y)
(al, ah), (ul, uh) = boot(pm, qm, y)
print(f'MEG-Mamba   acc {accuracy_score(y,pm):.3f} [{al:.3f}, {ah:.3f}]   '
      f'AUC {roc_auc_score(y,qm):.3f} [{ul:.3f}, {uh:.3f}]')

MEG-Mamba   acc 0.826 [0.764, 0.882]   AUC 0.890 [0.836, 0.940]


## Head to head against BrainOmni

Both models ran on the same 161 subjects, so this is a paired comparison. The parcellations
differ, Schaefer100 against Glasser52 or Brainnetome 210, which is a confound rather than a
clean model comparison. Worth stating that in the writeup.

The Glasser52 result is the fairest reference, since it is closest in channel count.

In [6]:
# Compare subject IDs, not labels. Labels are 77 ones then 84 zeros in both pipelines,
# so comparing them passes even if subjects were reordered within a group. The paired
# bootstrap below is only valid if row i is the same person in both feature sets.
ids_mamba = np.load(f'{OUT}/subject_ids_mamba.npy', allow_pickle=True)
ids_bo    = np.load(f'{PARCEL}/parcel_subject_ids_cortical.npy', allow_pickle=True)
assert np.array_equal(ids_mamba, ids_bo), (
    'subject order differs between the two pipelines, the paired test would be invalid')
print(f'subject order matches across both pipelines ({len(ids_bo)} subjects)')

refs = [('BrainOmni Glasser52 (52ch)',  f'{PARCEL}/parcel_frozen_tiny_glasser52.npy'),
        ('BrainOmni Brainnetome (210ch)', f'{PARCEL}/parcel_frozen_tiny_cortical.npy')]

print(f'{"method":32s} {"acc":>7s} {"AUC":>7s}')
print(f'{"MEG-Mamba Schaefer100 (100ch)":32s} {accuracy_score(y,pm):7.3f} {roc_auc_score(y,qm):7.3f}')
store = {}
for name, path in refs:
    if not os.path.exists(path):
        print(f'{name:32s}  missing'); continue
    p, q = oof(np.load(path), y)
    store[name] = (p, q)
    print(f'{name:32s} {accuracy_score(y,p):7.3f} {roc_auc_score(y,q):7.3f}')

print()
print('paired difference, MEG-Mamba minus BrainOmni, same subjects and resamples')
for name, (p, q) in store.items():
    rng, dA, dU = np.random.default_rng(123), [], []
    for _ in range(2000):
        b = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[b])) < 2:
            continue
        dA.append(accuracy_score(y[b], pm[b]) - accuracy_score(y[b], p[b]))
        dU.append(roc_auc_score(y[b], qm[b]) - roc_auc_score(y[b], q[b]))
    da = accuracy_score(y, pm) - accuracy_score(y, p)
    du = roc_auc_score(y, qm) - roc_auc_score(y, q)
    al2, ah2 = np.percentile(dA, [2.5, 97.5]); ul2, uh2 = np.percentile(dU, [2.5, 97.5])
    print(f'  vs {name}')
    print(f'     acc {da:+.3f} [{al2:+.3f}, {ah2:+.3f}]   '
          f'{"SIGNIFICANT" if al2 > 0 or ah2 < 0 else "not significant"}')
    print(f'     AUC {du:+.3f} [{ul2:+.3f}, {uh2:+.3f}]   '
          f'{"SIGNIFICANT" if ul2 > 0 or uh2 < 0 else "not significant"}')

subject order matches across both pipelines (161 subjects)
method                               acc     AUC
MEG-Mamba Schaefer100 (100ch)      0.826   0.890
BrainOmni Glasser52 (52ch)         0.870   0.904
BrainOmni Brainnetome (210ch)      0.851   0.913

paired difference, MEG-Mamba minus BrainOmni, same subjects and resamples
  vs BrainOmni Glasser52 (52ch)
     acc -0.043 [-0.099, +0.012]   not significant
     AUC -0.014 [-0.050, +0.023]   not significant
  vs BrainOmni Brainnetome (210ch)
     acc -0.025 [-0.081, +0.037]   not significant
     AUC -0.023 [-0.058, +0.010]   not significant
